# BINARY EXPERIMENT: INCREMENTAL VS FULL RETRAIN 100

In [16]:
import numpy as np
import pandas as pd
from pathlib import Path
from typing import Dict, Tuple, List
import re

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
)
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier

try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except Exception:
    HAS_XGB = False

# CONFIG

LABEL_COL = "Is_Suspicious"

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "4_class_balance").exists():
    parent = PROJECT_ROOT.parent
    if (parent / "4_class_balance").exists():
        PROJECT_ROOT = parent
    else:
        raise RuntimeError(f"Cannot find '4_class_balance' folder from cwd={Path.cwd()}")

VARIANT_ROOTS: Dict[str, Path] = {
    "baseline": PROJECT_ROOT / "4_class_balance" / "baseline_100",
    "smote":     PROJECT_ROOT / "4_class_balance" / "smote_100",
    "adasyn":     PROJECT_ROOT / "4_class_balance" / "adasyn_100",
}

VARIANT_TRAIN_FILE = {
    "baseline": "train_baseline.csv",
    "smote":    "train_smote.csv",
    "adasyn":   "train_adasyn.csv",
}

ALGOS = ["GBM", "XGB", "RF"]

N_FOLDS = 5          
TARGET_RECALL = 0.98
RANDOM_STATE = 42

WAVE_FILTER = None   


# =========================
# HELPERS
# =========================

def list_waves(root: Path) -> List[str]:
    if not root.exists():
        return []
    return sorted([p.name for p in root.iterdir() if p.is_dir()])


def _resolve_train_path(dir_path: Path, variant: str) -> Path:
    primary = dir_path / VARIANT_TRAIN_FILE[variant]
    if primary.exists():
        return primary
    fallback = dir_path / "train.csv"
    if fallback.exists():
        return fallback
    return primary


def load_frames(variant: str, wave: str) -> Tuple[pd.DataFrame, pd.DataFrame]:
    root = VARIANT_ROOTS[variant]
    d = root / wave
    train_path = _resolve_train_path(d, variant)
    test_path  = d / "test.csv"
    if not train_path.exists() or not test_path.exists():
        raise FileNotFoundError(
            f"Missing files for {variant}/{wave} → {train_path} / {test_path}"
        )
    tr = pd.read_csv(train_path)
    te = pd.read_csv(test_path)
    return tr, te


def wave_sort_key(wave: str):
    m = re.search(r"(\d+)$", str(wave))
    return int(m.group(1)) if m else wave


def make_model(algo: str, y_tr: np.ndarray):
    if algo == "GBM":
        return GradientBoostingClassifier(
            n_estimators=300,
            learning_rate=0.05,
            max_depth=3,
            random_state=RANDOM_STATE,
        )
    elif algo == "XGB" and HAS_XGB:
        pos = max(1, int((y_tr == 1).sum()))
        neg = max(1, int((y_tr == 0).sum()))
        spw = neg / pos
        return XGBClassifier(
            n_estimators=400,
            max_depth=4,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=0.8,
            reg_lambda=1.0,
            random_state=RANDOM_STATE,
            eval_metric="logloss",
            n_jobs=-1,
            tree_method="hist",
            scale_pos_weight=spw,
        )
    elif algo == "RF":
        return RandomForestClassifier(
            n_estimators=300,
            max_depth=None,
            min_samples_leaf=1,
            class_weight="balanced",
            random_state=RANDOM_STATE,
            n_jobs=-1,
        )
    else:
        raise RuntimeError("Unsupported algo or XGBoost not installed")


def make_sample_weights(y_tr: np.ndarray) -> np.ndarray:
    n_pos = max(1, int((y_tr == 1).sum()))
    n_neg = max(1, int((y_tr == 0).sum()))
    w_pos = n_neg / (n_pos + n_neg)
    w_neg = n_pos / (n_pos + n_neg)
    return np.where(y_tr == 1, w_pos, w_neg)


def predict_scores(model, X: np.ndarray) -> np.ndarray:
    if hasattr(model, "predict_proba"):
        p = model.predict_proba(X)[:, 1]
    else:
        try:
            dec = model.decision_function(X)
            p = (dec - dec.min()) / (dec.max() - dec.min() + 1e-12)
        except Exception:
            p = model.predict(X).astype(float)
    return p


def threshold_for_recall(y_true: np.ndarray, scores: np.ndarray, target: float):
    order = np.argsort(-scores)
    y_sorted = y_true[order]
    s_sorted = scores[order]
    P = int(y_true.sum())
    if P == 0:
        return 1.0, {
            "precision": 0.0,
            "recall": 0.0,
            "tp": 0,
            "fp": 0,
            "tn": int((y_true == 0).sum()),
            "fn": 0,
        }

    tp_cum = np.cumsum(y_sorted)
    fp_cum = np.cumsum(1 - y_sorted)
    recall_cum = tp_cum / (P + 1e-12)

    idx = np.where(recall_cum >= target)[0]
    if len(idx) == 0:
        thr = s_sorted[-1] - 1e-12
        pred = np.ones_like(y_true)
    else:
        k = int(idx[0])
        thr = s_sorted[k]
        pred = (scores >= thr).astype(int)

    tn, fp, fn, tp = confusion_matrix(y_true, pred, labels=[0, 1]).ravel()
    prec = tp / max(1, tp + fp)
    rec = tp / max(1, P)
    return float(thr), {
        "precision": float(prec),
        "recall": float(rec),
        "tp": int(tp),
        "fp": int(fp),
        "tn": int(tn),
        "fn": int(fn),
    }

# BINARY: INCREMENTAL VS FULL RETRAIN (B2…B5)

def run_binary_incremental_vs_fullretrain():
    """Binary experiment on 100-data: baseline/smote/adasyn, waves B2…B5."""
    print("[BINARY-100] Incremental vs Full retrain experiment")

    base_root = VARIANT_ROOTS["baseline"]
    waves_baseline = list_waves(base_root) or []
    if WAVE_FILTER is not None:
        waves_baseline = [w for w in waves_baseline if WAVE_FILTER(w)]
    waves_baseline = sorted(waves_baseline, key=wave_sort_key)
    assert len(waves_baseline) >= 2, "Need at least 2 waves for incremental/full retrain experiment."
    print(f"[baseline] detected {len(waves_baseline)} wave(s): {waves_baseline}")

    thr_col = f"thr@R>={TARGET_RECALL:.2f}"

    rows = []
    out_root = Path("./out_binary_incremental_100")
    out_root.mkdir(parents=True, exist_ok=True)

    for variant, root in VARIANT_ROOTS.items():
        print(f"\n=== VARIANT: {variant} ===")

        variant_waves = list_waves(root) or []
        if WAVE_FILTER is not None:
            variant_waves = [w for w in variant_waves if WAVE_FILTER(w)]
        variant_waves = sorted(variant_waves, key=wave_sort_key)

        if len(variant_waves) < 2:
            print(f"  Not enough waves for variant={variant}, skip.")
            continue

        wave_tr: Dict[str, pd.DataFrame] = {}
        wave_te: Dict[str, pd.DataFrame] = {}
        for wave in variant_waves:
            try:
                tr_df, te_df = load_frames(variant, wave)
            except FileNotFoundError as e:
                print(f"  [skip {variant}/{wave}] {e}")
                continue
            wave_tr[wave] = tr_df
            wave_te[wave] = te_df

        usable_waves = [w for w in variant_waves if w in wave_tr and w in wave_te]
        if len(usable_waves) < 2:
            print(f"  Not enough usable waves with data for variant={variant}, skip.")
            continue

        for idx in range(1, len(usable_waves)):
            wave_test = usable_waves[idx]
            wave_prev = usable_waves[idx - 1]

            print(f"\n  [WAVE] test={wave_test} (prev={wave_prev})")

            for algo in ALGOS:
                if algo == "XGB" and not HAS_XGB:
                    print("    [warn] xgboost not installed, skip XGB")
                    continue

                # Incremental
                train_waves_inc = [wave_prev]
                tr_inc = pd.concat([wave_tr[w] for w in train_waves_inc], ignore_index=True)

                y_tr_inc = tr_inc[LABEL_COL].astype(int).to_numpy()
                X_tr_inc = (
                    tr_inc.drop(columns=[LABEL_COL])
                    .select_dtypes(include=[np.number])
                    .to_numpy()
                )

                te_df = wave_te[wave_test]
                y_te = te_df[LABEL_COL].astype(int).to_numpy()
                X_te = (
                    te_df.drop(columns=[LABEL_COL])
                    .select_dtypes(include=[np.number])
                    .to_numpy()
                )

                model_inc = make_model(algo, y_tr_inc)
                sw_inc = make_sample_weights(y_tr_inc) if algo == "GBM" else None
                if sw_inc is not None:
                    model_inc.fit(X_tr_inc, y_tr_inc, sample_weight=sw_inc)
                else:
                    model_inc.fit(X_tr_inc, y_tr_inc)

                p_inc = predict_scores(model_inc, X_te)
                roc_inc = roc_auc_score(y_te, p_inc)
                prc_inc = average_precision_score(y_te, p_inc)
                thr_inc, at_inc = threshold_for_recall(y_te, p_inc, TARGET_RECALL)
                n_test = len(y_te)
                fp_per_1000_inc = at_inc["fp"] / max(1, n_test) * 1000.0

                print(
                    f"    [INCREMENTAL] algo={algo}, test_wave={wave_test}: "
                    f"ROC-AUC={roc_inc:.4f}, PR-AUC={prc_inc:.4f}, "
                    f"precision@R={at_inc['precision']:.3f}, recall={at_inc['recall']:.3f}, "
                    f"FP/1000={fp_per_1000_inc:.3f}"
                )

                rows.append(
                    {
                        "mode": "incremental",
                        "variant": variant,
                        "algo": algo,
                        "train_waves": ",".join(train_waves_inc),
                        "test_wave": wave_test,
                        "roc_auc": roc_inc,
                        "pr_auc": prc_inc,
                        thr_col: thr_inc,
                        "precision@R": at_inc["precision"],
                        "recall@thr": at_inc["recall"],
                        "TP": at_inc["tp"],
                        "FP": at_inc["fp"],
                        "TN": at_inc["tn"],
                        "FN": at_inc["fn"],
                        "FP_per_1000": fp_per_1000_inc,
                        "n_test": n_test,
                    }
                )

                # Full retrain
                train_waves_full = usable_waves[:idx]
                tr_full = pd.concat([wave_tr[w] for w in train_waves_full], ignore_index=True)

                y_tr_full = tr_full[LABEL_COL].astype(int).to_numpy()
                X_tr_full = (
                    tr_full.drop(columns=[LABEL_COL])
                    .select_dtypes(include=[np.number])
                    .to_numpy()
                )

                model_full = make_model(algo, y_tr_full)
                sw_full = make_sample_weights(y_tr_full) if algo == "GBM" else None
                if sw_full is not None:
                    model_full.fit(X_tr_full, y_tr_full, sample_weight=sw_full)
                else:
                    model_full.fit(X_tr_full, y_tr_full)

                p_full = predict_scores(model_full, X_te)
                roc_full = roc_auc_score(y_te, p_full)
                prc_full = average_precision_score(y_te, p_full)
                thr_full, at_full = threshold_for_recall(y_te, p_full, TARGET_RECALL)
                fp_per_1000_full = at_full["fp"] / max(1, n_test) * 1000.0

                print(
                    f"    [FULL]        algo={algo}, test_wave={wave_test}: "
                    f"ROC-AUC={roc_full:.4f}, PR-AUC={prc_full:.4f}, "
                    f"precision@R={at_full['precision']:.3f}, recall={at_full['recall']:.3f}, "
                    f"FP/1000={fp_per_1000_full:.3f}"
                )

                rows.append(
                    {
                        "mode": "full_retrain",
                        "variant": variant,
                        "algo": algo,
                        "train_waves": ",".join(train_waves_full),
                        "test_wave": wave_test,
                        "roc_auc": roc_full,
                        "pr_auc": prc_full,
                        thr_col: thr_full,
                        "precision@R": at_full["precision"],
                        "recall@thr": at_full["recall"],
                        "TP": at_full["tp"],
                        "FP": at_full["fp"],
                        "TN": at_full["tn"],
                        "FN": at_full["fn"],
                        "FP_per_1000": fp_per_1000_full,
                        "n_test": n_test,
                    }
                )

    if not rows:
        print("No rows produced for incremental/full retrain experiment.")
        return

    res = pd.DataFrame(rows)
    res.to_csv(out_root / "binary_incremental_fullretrain_by_wave_100.csv", index=False)

    def agg(df):
        return pd.Series(
            {
                "waves": len(df),
                "roc_auc_mean": df["roc_auc"].mean(),
                "pr_auc_mean": df["pr_auc"].mean(),
                "precision@R_mean": df["precision@R"].mean(),
                "recall@thr_mean": df["recall@thr"].mean(),
                "FP_per_1000_mean": df["FP_per_1000"].mean(),
            }
        )

    summary = res.groupby(["mode", "variant", "algo"], as_index=False).apply(agg)
    summary.to_csv(out_root / "binary_incremental_fullretrain_summary_100.csv", index=False)

if __name__ == "__main__":
    run_binary_incremental_vs_fullretrain()

[BINARY-100] Incremental vs Full retrain experiment
[baseline] detected 5 wave(s): ['synthetic_transactions_structured_100_1', 'synthetic_transactions_structured_100_2', 'synthetic_transactions_structured_100_3', 'synthetic_transactions_structured_100_4', 'synthetic_transactions_structured_100_5']

=== VARIANT: baseline ===

  [WAVE] test=synthetic_transactions_structured_100_2 (prev=synthetic_transactions_structured_100_1)
    [INCREMENTAL] algo=GBM, test_wave=synthetic_transactions_structured_100_2: ROC-AUC=0.9068, PR-AUC=0.7341, precision@R=0.014, recall=1.000, FP/1000=986.139
    [FULL]        algo=GBM, test_wave=synthetic_transactions_structured_100_2: ROC-AUC=0.9068, PR-AUC=0.7341, precision@R=0.014, recall=1.000, FP/1000=986.139
    [INCREMENTAL] algo=XGB, test_wave=synthetic_transactions_structured_100_2: ROC-AUC=0.9932, PR-AUC=0.8793, precision@R=0.169, recall=1.000, FP/1000=68.317
    [FULL]        algo=XGB, test_wave=synthetic_transactions_structured_100_2: ROC-AUC=0.9932, P

C:\Users\T470s\AppData\Local\Temp\ipykernel_19744\2178726426.py:381: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  summary = res.groupby(["mode", "variant", "algo"], as_index=False).apply(agg)


# OOF + PLATT SCALING + META MODEL (PER-WAVE FULL RETRAIN) 100

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from typing import Dict, Tuple, List
import re

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    brier_score_loss,
)
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except Exception:
    HAS_XGB = False


# === CONFIG ===
LABEL_COL = "Is_Suspicious"
SCENARIO_COL = "Suspicion Category"

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "4_class_balance").exists():
    parent = PROJECT_ROOT.parent
    if (parent / "4_class_balance").exists():
        PROJECT_ROOT = parent
    else:
        raise RuntimeError(f"Cannot find '4_class_balance' folder from cwd={Path.cwd()}")

VARIANT_ROOTS: Dict[str, Path] = {
    "baseline": PROJECT_ROOT / "4_class_balance" / "baseline_100",
    "smote":     PROJECT_ROOT / "4_class_balance" / "smote_100",
    "adasyn":     PROJECT_ROOT / "4_class_balance" / "adasyn_100",
}

VARIANT_TRAIN_FILE = {
    "baseline": "train_baseline.csv",
    "smote":    "train_smote.csv",
    "adasyn":   "train_adasyn.csv",
}

ALGOS = ["GBM", "XGB", "RF"]

TARGET_RECALL = 0.98
RANDOM_STATE = 42
WAVE_FILTER = None

# Calibration bins settings
CALIB_BINS = 10
CALIB_STRATEGY = "quantile"  # "quantile" or "uniform"


# === HELPERS ===
def list_waves(root: Path) -> List[str]:
    if not root.exists():
        return []
    return sorted([p.name for p in root.iterdir() if p.is_dir()])


def _resolve_train_path(dir_path: Path, variant: str) -> Path:
    primary = dir_path / VARIANT_TRAIN_FILE[variant]
    if primary.exists():
        return primary
    fallback = dir_path / "train.csv"
    if fallback.exists():
        return fallback
    return primary


def load_frames(variant: str, wave: str) -> Tuple[pd.DataFrame, pd.DataFrame]:
    root = VARIANT_ROOTS[variant]
    d = root / wave
    train_path = _resolve_train_path(d, variant)
    test_path = d / "test.csv"
    if not train_path.exists() or not test_path.exists():
        raise FileNotFoundError(f"Missing files for {variant}/{wave} → {train_path} / {test_path}")
    tr = pd.read_csv(train_path)
    te = pd.read_csv(test_path)
    return tr, te


def wave_sort_key(wave: str):
    m = re.search(r"(\d+)$", str(wave))
    return int(m.group(1)) if m else wave


# === MODELS ===
def make_model(algo: str, y_tr: np.ndarray):
    if algo == "GBM":
        return GradientBoostingClassifier(
            n_estimators=300,
            learning_rate=0.05,
            max_depth=3,
            random_state=RANDOM_STATE,
        )
    elif algo == "XGB" and HAS_XGB:
        pos = max(1, int((y_tr == 1).sum()))
        neg = max(1, int((y_tr == 0).sum()))
        spw = neg / pos
        return XGBClassifier(
            n_estimators=400,
            max_depth=4,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=0.8,
            reg_lambda=1.0,
            random_state=RANDOM_STATE,
            eval_metric="logloss",
            n_jobs=-1,
            tree_method="hist",
            scale_pos_weight=spw,
        )
    elif algo == "RF":
        return RandomForestClassifier(
            n_estimators=300,
            max_depth=None,
            min_samples_leaf=1,
            class_weight="balanced",
            random_state=RANDOM_STATE,
            n_jobs=-1,
        )
    else:
        raise RuntimeError("Unsupported algo or XGBoost not installed")


def make_sample_weights(y_tr: np.ndarray) -> np.ndarray:
    n_pos = max(1, int((y_tr == 1).sum()))
    n_neg = max(1, int((y_tr == 0).sum()))
    w_pos = n_neg / (n_pos + n_neg)
    w_neg = n_pos / (n_pos + n_neg)
    return np.where(y_tr == 1, w_pos, w_neg)


def predict_scores(model, X: np.ndarray) -> np.ndarray:
    if hasattr(model, "predict_proba"):
        return model.predict_proba(X)[:, 1]
    try:
        dec = model.decision_function(X)
        return (dec - dec.min()) / (dec.max() - dec.min() + 1e-12)
    except Exception:
        return model.predict(X).astype(float)


def threshold_for_recall(y_true: np.ndarray, scores: np.ndarray, target: float):
    order = np.argsort(-scores)
    y_sorted = y_true[order]
    s_sorted = scores[order]
    P = int(y_true.sum())
    if P == 0:
        return 1.0, {"precision": 0.0, "recall": 0.0, "tp": 0, "fp": 0, "tn": int((y_true == 0).sum()), "fn": 0}

    tp_cum = np.cumsum(y_sorted)
    recall_cum = tp_cum / (P + 1e-12)

    idx = np.where(recall_cum >= target)[0]
    if len(idx) == 0:
        thr = s_sorted[-1] - 1e-12
    else:
        thr = s_sorted[int(idx[0])]

    pred = (scores >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, pred, labels=[0, 1]).ravel()
    prec = tp / max(1, tp + fp)
    rec = tp / max(1, P)
    return float(thr), {"precision": float(prec), "recall": float(rec), "tp": int(tp), "fp": int(fp), "tn": int(tn), "fn": int(fn)}


def _feature_cols_from_train(tr_df: pd.DataFrame) -> List[str]:
    drop = {LABEL_COL}
    if SCENARIO_COL in tr_df.columns:
        drop.add(SCENARIO_COL)
    return [c for c in tr_df.columns if c not in drop]


def _build_Xy(tr_df: pd.DataFrame, te_df: pd.DataFrame):
    feat_cols = _feature_cols_from_train(tr_df)
    X_tr = tr_df[feat_cols].to_numpy()
    y_tr = tr_df[LABEL_COL].astype(int).to_numpy()
    X_te = te_df.reindex(columns=feat_cols, fill_value=0).to_numpy()
    y_te = te_df[LABEL_COL].astype(int).to_numpy()
    return X_tr, y_tr, X_te, y_te, feat_cols


def _safe_train_val_split(X: np.ndarray, y: np.ndarray, test_size: float = 0.2):
    n_pos = int((y == 1).sum())
    n_neg = int((y == 0).sum())
    if len(y) < 10 or n_pos < 2 or n_neg < 2:
        return X, X, y, y
    return train_test_split(X, y, test_size=test_size, random_state=RANDOM_STATE, stratify=y)


def _metrics_at_threshold(y_true: np.ndarray, scores: np.ndarray, thr: float):
    pred = (scores >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, pred, labels=[0, 1]).ravel()
    P = int(y_true.sum())
    prec = tp / max(1, tp + fp)
    rec = tp / max(1, P)
    return {"precision": float(prec), "recall": float(rec), "tp": int(tp), "fp": int(fp), "tn": int(tn), "fn": int(fn)}


# === CALIBRATION (Platt + reliability table) ===
def fit_platt(scores: np.ndarray, y: np.ndarray) -> LogisticRegression:
    lr = LogisticRegression(solver="lbfgs", max_iter=1000)
    lr.fit(scores.reshape(-1, 1), y)
    return lr


def apply_platt(lr: LogisticRegression, scores: np.ndarray) -> np.ndarray:
    return lr.predict_proba(scores.reshape(-1, 1))[:, 1]


def calibration_bins_table(y: np.ndarray, p: np.ndarray, n_bins: int = 10, strategy: str = "quantile") -> Tuple[pd.DataFrame, float]:
    y = y.astype(int)
    p = np.clip(p.astype(float), 0.0, 1.0)
    n = len(y)

    if n == 0:
        df = pd.DataFrame(columns=["bin", "count", "mean_pred", "frac_pos", "abs_gap"])
        return df, np.nan

    if strategy == "quantile":
        qs = np.linspace(0, 1, n_bins + 1)
        edges = np.quantile(p, qs)
        edges[0] = 0.0
        edges[-1] = 1.0
        edges = np.unique(edges)
        if len(edges) < 3:
            edges = np.linspace(0, 1, n_bins + 1)
    else:
        edges = np.linspace(0, 1, n_bins + 1)

    bin_ids = np.digitize(p, edges[1:-1], right=True)
    rows = []
    ece = 0.0

    for b in range(len(edges) - 1):
        mask = bin_ids == b
        cnt = int(mask.sum())
        if cnt == 0:
            continue
        mean_pred = float(p[mask].mean())
        frac_pos = float(y[mask].mean())
        abs_gap = abs(frac_pos - mean_pred)
        ece += (cnt / n) * abs_gap
        rows.append({
            "bin": b,
            "count": cnt,
            "edge_left": float(edges[b]),
            "edge_right": float(edges[b + 1]),
            "mean_pred": mean_pred,
            "frac_pos": frac_pos,
            "abs_gap": abs_gap
        })

    df = pd.DataFrame(rows)
    return df, float(ece)


# === MAIN EXPERIMENT ===
def run_binary_incremental_vs_fullretrain_with_calibration():
    print("[BINARY] Incremental vs Full retrain + Calibration (Raw vs Platt)")
    base_root = VARIANT_ROOTS["baseline"]
    waves = list_waves(base_root) or []
    if WAVE_FILTER is not None:
        waves = [w for w in waves if WAVE_FILTER(w)]
    waves = sorted(waves, key=wave_sort_key)
    assert len(waves) >= 2, "Need at least 2 waves."
    print(f"Detected {len(waves)} wave(s): {waves}")

    thr_col = f"thr@R>={TARGET_RECALL:.2f}"

    rows = []
    out_root = PROJECT_ROOT / "6_training_mode" / "out_binary_incremental_100_calibration"
    out_root.mkdir(parents=True, exist_ok=True)

    pred_dir = out_root / "predictions"
    bins_dir = out_root / "calibration_bins"
    pred_dir.mkdir(parents=True, exist_ok=True)
    bins_dir.mkdir(parents=True, exist_ok=True)

    for variant in VARIANT_ROOTS.keys():  # baseline, rose
        print(f"\n=== VARIANT: {variant} ===")

        wave_tr: Dict[str, pd.DataFrame] = {}
        wave_te: Dict[str, pd.DataFrame] = {}
        for wave in waves:
            try:
                tr_df, te_df = load_frames(variant, wave)
            except FileNotFoundError as e:
                print(f"  [skip {variant}/{wave}] {e}")
                continue
            wave_tr[wave] = tr_df
            wave_te[wave] = te_df

        usable_waves = [w for w in waves if w in wave_tr and w in wave_te]
        if len(usable_waves) < 2:
            print(f"  Not enough waves for variant={variant}, skip.")
            continue

        for idx in range(1, len(usable_waves)):
            wave_test = usable_waves[idx]
            wave_prev = usable_waves[idx - 1]
            te_df = wave_te[wave_test]

            print(f"\n  [WAVE] test={wave_test} (prev={wave_prev})")

            for algo in ALGOS:
                if algo == "XGB" and not HAS_XGB:
                    print("    [warn] xgboost not installed, skip XGB")
                    continue

                for mode in ["incremental", "full_retrain"]:
                    if mode == "incremental":
                        train_waves = [wave_prev]
                    else:
                        train_waves = usable_waves[:idx]

                    tr_df = pd.concat([wave_tr[w] for w in train_waves], ignore_index=True)

                    X_tr, y_tr, X_te, y_te, _ = _build_Xy(tr_df, te_df)
                    X_fit, X_val, y_fit, y_val = _safe_train_val_split(X_tr, y_tr)

                    model = make_model(algo, y_fit)
                    sw = make_sample_weights(y_fit) if algo == "GBM" else None
                    if sw is not None:
                        model.fit(X_fit, y_fit, sample_weight=sw)
                    else:
                        model.fit(X_fit, y_fit)

                    # ---- RAW scores ----
                    p_val_raw = predict_scores(model, X_val)
                    p_te_raw = predict_scores(model, X_te)

                    roc_raw = roc_auc_score(y_te, p_te_raw)
                    prc_raw = average_precision_score(y_te, p_te_raw)

                    thr_raw, _ = threshold_for_recall(y_val, p_val_raw, TARGET_RECALL)
                    at_raw = _metrics_at_threshold(y_te, p_te_raw, thr_raw)

                    brier_raw = brier_score_loss(y_te, np.clip(p_te_raw, 0, 1))
                    bins_raw, ece_raw = calibration_bins_table(
                        y_te, p_te_raw, n_bins=CALIB_BINS, strategy=CALIB_STRATEGY
                    )

                    # ---- PLATT (fit on validation predictions) ----
                    platt_ok = (len(np.unique(y_val)) == 2) and (len(y_val) >= 10)
                    if platt_ok:
                        platt = fit_platt(p_val_raw, y_val)
                        p_val_platt = apply_platt(platt, p_val_raw)
                        p_te_platt = apply_platt(platt, p_te_raw)

                        roc_platt = roc_auc_score(y_te, p_te_platt)
                        prc_platt = average_precision_score(y_te, p_te_platt)

                        thr_platt, _ = threshold_for_recall(y_val, p_val_platt, TARGET_RECALL)
                        at_platt = _metrics_at_threshold(y_te, p_te_platt, thr_platt)

                        brier_platt = brier_score_loss(y_te, np.clip(p_te_platt, 0, 1))
                        bins_platt, ece_platt = calibration_bins_table(
                            y_te, p_te_platt, n_bins=CALIB_BINS, strategy=CALIB_STRATEGY
                        )
                    else:
                        # fallback: no calibration possible (too few positives in val)
                        p_te_platt = np.full_like(p_te_raw, np.nan, dtype=float)
                        roc_platt = np.nan
                        prc_platt = np.nan
                        thr_platt = np.nan
                        at_platt = {"precision": np.nan, "recall": np.nan, "tp": np.nan, "fp": np.nan, "tn": np.nan, "fn": np.nan}
                        brier_platt = np.nan
                        ece_platt = np.nan
                        bins_platt = pd.DataFrame()

                    n_test = len(y_te)
                    fp1000_raw = at_raw["fp"] / max(1, n_test) * 1000.0
                    fp1000_platt = (at_platt["fp"] / max(1, n_test) * 1000.0) if platt_ok else np.nan

                    print(
                        f"    [{mode.upper():11s}] algo={algo} | RAW: "
                        f"ROC-AUC={roc_raw:.4f}, PR-AUC={prc_raw:.4f}, "
                        f"Prec@R={at_raw['precision']:.3f}, R={at_raw['recall']:.3f}, "
                        f"FP/1000={fp1000_raw:.3f}, Brier={brier_raw:.6f}, ECE={ece_raw:.4f}"
                    )
                    if platt_ok:
                        print(
                            f"                 algo={algo} | PLATT: "
                            f"ROC-AUC={roc_platt:.4f}, PR-AUC={prc_platt:.4f}, "
                            f"Prec@R={at_platt['precision']:.3f}, R={at_platt['recall']:.3f}, "
                            f"FP/1000={fp1000_platt:.3f}, Brier={brier_platt:.6f}, ECE={ece_platt:.4f}"
                        )
                    else:
                        print("                 PLATT skipped (validation set has too few positives/negatives).")

                    # Save predictions for later plots
                    pred_df = pd.DataFrame({
                        "y_true": y_te.astype(int),
                        "p_raw": p_te_raw,
                        "p_platt": p_te_platt,
                        "pred_raw": (p_te_raw >= thr_raw).astype(int),
                        "pred_platt": (p_te_platt >= thr_platt).astype(int) if platt_ok else np.nan,
                    })
                    pred_path = pred_dir / f"pred_{variant}_{mode}_{algo}_{wave_test}.csv"
                    pred_df.to_csv(pred_path, index=False)

                    # Save calibration bins
                    bins_raw.to_csv(bins_dir / f"calib_bins_RAW_{variant}_{mode}_{algo}_{wave_test}.csv", index=False)
                    if platt_ok:
                        bins_platt.to_csv(bins_dir / f"calib_bins_PLATT_{variant}_{mode}_{algo}_{wave_test}.csv", index=False)

                    # Store row
                    rows.append({
                        "mode": mode,
                        "variant": variant,
                        "algo": algo,
                        "train_waves": ",".join(train_waves),
                        "test_wave": wave_test,

                        # RAW
                        "roc_auc_raw": roc_raw,
                        "pr_auc_raw": prc_raw,
                        f"{thr_col}_raw": thr_raw,
                        "precision@R_raw": at_raw["precision"],
                        "recall@thr_raw": at_raw["recall"],
                        "TP_raw": at_raw["tp"],
                        "FP_raw": at_raw["fp"],
                        "TN_raw": at_raw["tn"],
                        "FN_raw": at_raw["fn"],
                        "FP_per_1000_raw": fp1000_raw,
                        "brier_raw": brier_raw,
                        "ece_raw": ece_raw,

                        # PLATT
                        "roc_auc_platt": roc_platt,
                        "pr_auc_platt": prc_platt,
                        f"{thr_col}_platt": thr_platt,
                        "precision@R_platt": at_platt["precision"],
                        "recall@thr_platt": at_platt["recall"],
                        "TP_platt": at_platt["tp"],
                        "FP_platt": at_platt["fp"],
                        "TN_platt": at_platt["tn"],
                        "FN_platt": at_platt["fn"],
                        "FP_per_1000_platt": fp1000_platt,
                        "brier_platt": brier_platt,
                        "ece_platt": ece_platt,

                        "n_test": n_test,
                        "calibration_strategy": CALIB_STRATEGY,
                        "calibration_bins": CALIB_BINS,
                        "platt_used": bool(platt_ok),
                    })

    if not rows:
        print("No rows produced.")
        return

    res = pd.DataFrame(rows)
    res.to_csv(out_root / "binary_incremental_fullretrain_by_wave_with_calibration.csv", index=False)

    def agg(df):
        return pd.Series({
            "waves": len(df),

            "roc_auc_raw_mean": df["roc_auc_raw"].mean(),
            "pr_auc_raw_mean": df["pr_auc_raw"].mean(),
            "precision@R_raw_mean": df["precision@R_raw"].mean(),
            "FP_per_1000_raw_mean": df["FP_per_1000_raw"].mean(),
            "brier_raw_mean": df["brier_raw"].mean(),
            "ece_raw_mean": df["ece_raw"].mean(),

            "roc_auc_platt_mean": df["roc_auc_platt"].mean(),
            "pr_auc_platt_mean": df["pr_auc_platt"].mean(),
            "precision@R_platt_mean": df["precision@R_platt"].mean(),
            "FP_per_1000_platt_mean": df["FP_per_1000_platt"].mean(),
            "brier_platt_mean": df["brier_platt"].mean(),
            "ece_platt_mean": df["ece_platt"].mean(),

            # improvements (negative = better for Brier/ECE/FP)
            "delta_brier_mean": (df["brier_platt"] - df["brier_raw"]).mean(),
            "delta_ece_mean": (df["ece_platt"] - df["ece_raw"]).mean(),
            "delta_FP1000_mean": (df["FP_per_1000_platt"] - df["FP_per_1000_raw"]).mean(),
            "delta_precision@R_mean": (df["precision@R_platt"] - df["precision@R_raw"]).mean(),
        })

    summary = res.groupby(["mode", "variant", "algo"], as_index=False).apply(agg)
    summary.to_csv(out_root / "binary_incremental_fullretrain_summary_with_calibration.csv", index=False)

    print("\nSaved:")
    print(" -", out_root / "binary_incremental_fullretrain_by_wave_with_calibration.csv")
    print(" -", out_root / "binary_incremental_fullretrain_summary_with_calibration.csv")
    print(" - predictions in:", pred_dir)
    print(" - calibration bins in:", bins_dir)


if __name__ == "__main__":
    run_binary_incremental_vs_fullretrain_with_calibration()


[BINARY] Incremental vs Full retrain + Calibration (Raw vs Platt)
Detected 5 wave(s): ['synthetic_transactions_structured_100_1', 'synthetic_transactions_structured_100_2', 'synthetic_transactions_structured_100_3', 'synthetic_transactions_structured_100_4', 'synthetic_transactions_structured_100_5']

=== VARIANT: baseline ===

  [WAVE] test=synthetic_transactions_structured_100_2 (prev=synthetic_transactions_structured_100_1)
    [INCREMENTAL] algo=GBM | RAW: ROC-AUC=0.9703, PR-AUC=0.7512, Prec@R=0.463, R=0.679, FP/1000=10.891, Brier=0.013467, ECE=0.0127
                 algo=GBM | PLATT: ROC-AUC=0.9703, PR-AUC=0.7512, Prec@R=0.463, R=0.679, FP/1000=10.891, Brier=0.007685, ECE=0.0070
    [FULL_RETRAIN] algo=GBM | RAW: ROC-AUC=0.9703, PR-AUC=0.7512, Prec@R=0.463, R=0.679, FP/1000=10.891, Brier=0.013467, ECE=0.0127
                 algo=GBM | PLATT: ROC-AUC=0.9703, PR-AUC=0.7512, Prec@R=0.463, R=0.679, FP/1000=10.891, Brier=0.007685, ECE=0.0070
    [INCREMENTAL] algo=XGB | RAW: ROC-AUC=

# MULTI-CLASS MODEL (SCENARIO CLASSIFICATION) 100

In [21]:
import numpy as np
import pandas as pd
from pathlib import Path
from typing import List

from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    confusion_matrix,
    classification_report,
)

# CONFIG
SCENARIO_COL = "Suspicion Category"

# Folder with OOF tables from binary OOF script on 100-data
OOF_ROOT = Path("./out_oof_meta_100")

VARIANTS: List[str] = ["baseline"]
RANDOM_STATE = 42

# HELPERS
def list_waves_from_oof(variant: str) -> List[str]:
    """List waves for a variant based on folders."""
    base = OOF_ROOT / variant
    if not base.exists():
        return []
    return sorted([p.name for p in base.iterdir() if p.is_dir()])

def load_oof_tables(variant: str, wave: str):
    """Load train/test OOF tables for a wave."""
    base = OOF_ROOT / variant / wave
    tr_path = base / "train_with_oof.csv"
    te_path = base / "test_with_pred.csv"
    if not tr_path.exists() or not te_path.exists():
        raise FileNotFoundError(
            f"Missing OOF files for {variant}/{wave}: {tr_path} / {te_path}"
        )
    tr = pd.read_csv(tr_path)
    te = pd.read_csv(te_path)
    return tr, te

# MULTI-CLASS META MODEL
def run_multiclass_meta():
    """Train and evaluate multi-class model on OOF scores."""
    rows = []

    for variant in VARIANTS:
        waves = list_waves_from_oof(variant)
        if not waves:
            print(f"[skip] No waves found for variant={variant}")
            continue

        print(f"\n=== VARIANT: {variant} ===")
        for wave in waves:
            try:
                tr, te = load_oof_tables(variant, wave)
            except FileNotFoundError as e:
                print(f"  [skip {variant}/{wave}] {e}")
                continue

            if SCENARIO_COL not in tr.columns or SCENARIO_COL not in te.columns:
                print(f"  [skip {variant}/{wave}] no '{SCENARIO_COL}' in tables")
                continue

            # scenario labels as strings
            scen_tr = tr[SCENARIO_COL].fillna("Normal").astype(str)
            scen_te = te[SCENARIO_COL].fillna("Normal").astype(str)

            # only suspicious transactions (scenario != "Normal")
            mask_tr = scen_tr != "Normal"
            mask_te = scen_te != "Normal"

            if mask_tr.sum() == 0 or mask_te.sum() == 0:
                print(f"  [skip {variant}/{wave}] no suspicious rows for multi-class")
                continue

            # OOF features from binary models
            oof_cols = [c for c in tr.columns if c.startswith("oof_") and c.endswith("_bin")]
            if not oof_cols:
                print(f"  [skip {variant}/{wave}] no OOF columns (oof_*_bin)")
                continue

            pred_cols = [c.replace("oof_", "pred_") for c in oof_cols]
            missing_pred = [c for c in pred_cols if c not in te.columns]
            if missing_pred:
                print(f"  [skip {variant}/{wave}] missing test preds: {missing_pred}")
                continue

            # base features: same as for binary model (exclude labels + OOF/pred columns)
            base_cols = [
                c for c in tr.columns
                if c not in ("Is_Suspicious", SCENARIO_COL)
                and not c.startswith("oof_")
                and not c.startswith("pred_")
            ]

            # stacking features = original features + OOF features
            train_cols = base_cols + oof_cols
            test_cols  = base_cols + pred_cols

            X_tr = tr.loc[mask_tr, train_cols].to_numpy()
            X_te = te.loc[mask_te, test_cols].to_numpy()
            y_tr_labels = scen_tr[mask_tr].to_numpy()
            y_te_labels = scen_te[mask_te].to_numpy()

            print(
                f"  [WAVE] {wave}: "
                f"suspicious train={X_tr.shape[0]}, suspicious test={X_te.shape[0]}"
            )

            # encode scenario labels to integers
            le = LabelEncoder()
            le.fit(np.concatenate([y_tr_labels, y_te_labels]))
            y_tr = le.transform(y_tr_labels)
            y_te = le.transform(y_te_labels)

            # multi-class classifier
            clf = LogisticRegression(
                multi_class="multinomial",
                max_iter=1000,
                random_state=RANDOM_STATE,
            )
            clf.fit(X_tr, y_tr)
            y_pred = clf.predict(X_te)

            acc = accuracy_score(y_te, y_pred)
            f1_macro = f1_score(y_te, y_pred, average="macro")
            prec_macro = precision_score(
                y_te, y_pred, average="macro", zero_division=0
            )
            rec_macro = recall_score(
                y_te, y_pred, average="macro", zero_division=0
            )

            print(
                f"    [MULTI-CLASS TEST] acc={acc:.4f}, F1_macro={f1_macro:.4f}, "
                f"precision_macro={prec_macro:.4f}, recall_macro={rec_macro:.4f}"
            )

            rows.append(
                {
                    "variant": variant,
                    "wave": wave,
                    "n_train_suspicious": int(mask_tr.sum()),
                    "n_test_suspicious": int(mask_te.sum()),
                    "accuracy": float(acc),
                    "f1_macro": float(f1_macro),
                    "precision_macro": float(prec_macro),
                    "recall_macro": float(rec_macro),
                }
            )

            out_dir = OOF_ROOT / variant / wave
            out_dir.mkdir(parents=True, exist_ok=True)

            # confusion matrix with proper labels
            classes_int = np.arange(len(le.classes_))
            class_names = list(le.classes_)
            cm = confusion_matrix(y_te, y_pred, labels=classes_int)
            cm_df = pd.DataFrame(cm, index=class_names, columns=class_names)
            cm_df.index.name = "true"
            cm_df.columns.name = "pred"
            cm_df.to_csv(out_dir / "scenario_confusion_matrix.csv")

            # per-class metrics with names
            rep = classification_report(
                y_te,
                y_pred,
                labels=classes_int,
                target_names=class_names,
                output_dict=True,
                zero_division=0,
            )
            rep_df = pd.DataFrame(rep).transpose()
            rep_df.to_csv(out_dir / "scenario_classification_report.csv")

            # save test with scenario predictions (as names)
            te_out = te.copy()
            te_out["Scenario_Pred"] = "Normal"
            te_out.loc[mask_te, "Scenario_Pred"] = le.inverse_transform(y_pred)
            te_out.to_csv(out_dir / "test_with_scenario_pred.csv", index=False)

    if not rows:
        print("No rows produced for multi-class meta experiment.")
        return

    res = pd.DataFrame(rows)
    out_path = OOF_ROOT / "multiclass_meta_summary.csv"
    res.to_csv(out_path, index=False)
    print(f"\nSaved multi-class summary to: {out_path}")

if __name__ == "__main__":
    run_multiclass_meta()


=== VARIANT: baseline ===
  [WAVE] synthetic_transactions_structured_100_1: suspicious train=65, suspicious test=35
    [MULTI-CLASS TEST] acc=0.6857, F1_macro=0.5833, precision_macro=0.6667, recall_macro=0.5357


C:\Users\T470s\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\linear_model\_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


  [WAVE] synthetic_transactions_structured_100_2: suspicious train=72, suspicious test=28
    [MULTI-CLASS TEST] acc=0.8929, F1_macro=0.8631, precision_macro=0.8939, recall_macro=0.8712


C:\Users\T470s\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\linear_model\_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


  [WAVE] synthetic_transactions_structured_100_3: suspicious train=80, suspicious test=20
    [MULTI-CLASS TEST] acc=1.0000, F1_macro=1.0000, precision_macro=1.0000, recall_macro=1.0000


...

  [WAVE] synthetic_transactions_structured_100_4: suspicious train=71, suspicious test=29
    [MULTI-CLASS TEST] acc=0.7931, F1_macro=0.7630, precision_macro=0.8185, recall_macro=0.8056


C:\Users\T470s\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\linear_model\_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


  [WAVE] synthetic_transactions_structured_100_5: suspicious train=70, suspicious test=30
    [MULTI-CLASS TEST] acc=1.0000, F1_macro=1.0000, precision_macro=1.0000, recall_macro=1.0000


C:\Users\T470s\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\linear_model\_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(



Saved multi-class summary to: out_oof_meta_100\multiclass_meta_summary.csv
